In [ ]:
###############################################################################
# C0: COMPLETE SETUP — Option A Bottom-Up Analysis
# GSE182159 (243K cells, 59 subclusters, 23 donors, 5 groups)
# 2026-03-01
# 바닥부터: proportion → pathway → gene → pattern → theory
###############################################################################


# =================================================================
# CELL 1: Install & Import
# =================================================================
!pip install scanpy anndata matplotlib seaborn scipy -q

import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.sparse import issparse
import os, warnings
warnings.filterwarnings('ignore')

print(f"scanpy:  {sc.__version__}")
print(f"anndata: {ad.__version__}")
print(f"pandas:  {pd.__version__}")
print(f"numpy:   {np.__version__}")
print("✅ Cell 1 complete")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 100.3 MB/s eta 0:00:00


In [ ]:
# =================================================================
# CELL 2: Google Drive Mount + Output Directory
# =================================================================
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/ITLAS'
DATA_DIR = f'{BASE}/data/processed'
RESULTS  = f'{BASE}/results'

# Fresh output — Option A
OUT = f'{RESULTS}/optionA_26Mar01'
os.makedirs(OUT, exist_ok=True)

# Sub-folders per analysis step
for d in ['C1_proportion', 'C2_pathway', 'C3_gene', 'C4_pattern',
          'C5_transition', 'C6_synthesis', 'tables']:
    os.makedirs(f'{OUT}/{d}', exist_ok=True)

print(f"✅ Cell 2 complete — output: {OUT}")

Mounted at /content/drive
✅ Cell 2 complete — output: /content/drive/MyDrive/ITLAS/results/optionA_26Mar01


In [ ]:
# =================================================================
# CELL 3: Load h5ad
# =================================================================
import time
t0 = time.time()

H5AD = f'{DATA_DIR}/GSE182159_gut2021_annotated.h5ad'
if not os.path.exists(H5AD):
    # fallback
    H5AD = f'{BASE}/data/GSE182159_gut2021_annotated.h5ad'

print(f"Loading: {H5AD}")
adata = sc.read_h5ad(H5AD)
print(f"Shape: {adata.shape[0]:,} cells × {adata.shape[1]:,} genes")
print(f"Loaded in {time.time()-t0:.1f}s")
print("✅ Cell 3 complete")


Loading: /content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad
Shape: 243,000 cells × 24,452 genes
Loaded in 301.1s
✅ Cell 3 complete


In [ ]:
# =================================================================
# CELL 4: Column Discovery (전체 구조 파악)
# =================================================================
print("=" * 70)
print("  adata.obs columns")
print("=" * 70)
for i, col in enumerate(adata.obs.columns):
    n = adata.obs[col].nunique()
    print(f"  [{i:2d}] {col:45s} nunique={n}")

print(f"\n  obsm: {list(adata.obsm.keys())}")
print(f"  uns:  {list(adata.uns.keys())[:8]}")
print("✅ Cell 4 complete")

  adata.obs columns
  [ 0] sample                                        nunique=46
  [ 1] tissue                                        nunique=2
  [ 2] Stage                                         nunique=5
  [ 3] IT_cluster_21                                 nunique=242960
  [ 4] IT_cluster_23                                 nunique=240984
  [ 5] IT_cluster_25                                 nunique=114299
  [ 6] IT_nk_collapse                                nunique=168875
  [ 7] IT_IT_signature                               nunique=243000
  [ 8] GSM_ID                                        nunique=46
  [ 9] IT_score_v2                                   nunique=243000
  [10] IT_score_v3                                   nunique=243000
  [11] IT_score_v4                                   nunique=242998
  [12] IT_signature_final                            nunique=242998
  [13] IT_like                                       nunique=2
  [14] PW_mTOR_signaling                           

In [ ]:
# =================================================================
# CELL 5: Key Column Assignment + Verification
# =================================================================

# ---- (A) Stage column ----
STAGE_COL = None
for c in ['Stage', 'stage', 'disease_stage']:
    if c in adata.obs.columns:
        STAGE_COL = c; break
assert STAGE_COL, "❌ Stage column not found"

stages = sorted(adata.obs[STAGE_COL].unique().tolist())
print(f"STAGE_COL = '{STAGE_COL}' → {stages}")
assert set(['NL','IT','IA','AR','CR']) == set(stages), f"Unexpected stages: {stages}"

# ---- (B) Subcluster column ----
SUB_COL = None
for c in ['gut2021_subcluster_v2', 'subcluster', 'celltype']:
    if c in adata.obs.columns:
        SUB_COL = c; break
assert SUB_COL, "❌ Subcluster column not found"

n_sub = adata.obs[SUB_COL].nunique()
print(f"SUB_COL   = '{SUB_COL}' → {n_sub} subclusters")

# ---- (C) Sample column ----
SAMPLE_COL = None
for c in ['sample', 'sample_id', 'Sample']:
    if c in adata.obs.columns:
        SAMPLE_COL = c; break
assert SAMPLE_COL, "❌ Sample column not found"
print(f"SAMPLE_COL= '{SAMPLE_COL}'")

# Sample format 확인
print(f"\n  Sample examples:")
for s in adata.obs[SAMPLE_COL].unique()[:5]:
    print(f"    {s}")

print("\n✅ Cell 5 complete")

STAGE_COL = 'Stage' → ['AR', 'CR', 'IA', 'IT', 'NL']
SUB_COL   = 'gut2021_subcluster_v2' → 59 subclusters
SAMPLE_COL= 'sample'

  Sample examples:
    GSM5519467_P190604_Blood_1
    GSM5519468_P190604_Blood_2
    GSM5519469_P190604_Liver_1
    GSM5519470_P190326_Blood_1
    GSM5519471_P190326_Liver_1

✅ Cell 5 complete


In [ ]:
# =================================================================
# CELL 6: Lineage Assignment
# =================================================================
# (1) major_lineage column 이 이미 있으면 활용
# (2) 없으면 subcluster prefix에서 추출

LINEAGE_COL = 'lineage'  # 우리가 만들 표준 column

if 'major_lineage' in adata.obs.columns:
    print("✅ 'major_lineage' column exists — mapping to standard names")
    raw_vals = sorted(adata.obs['major_lineage'].unique().tolist())
    print(f"   Raw values: {raw_vals}")

    lmap = {}
    for v in raw_vals:
        s = str(v)
        if 'CD4' in s:                          lmap[v] = 'CD4T'
        elif 'CD8' in s:                         lmap[v] = 'CD8T'
        elif 'gd' in s.lower() or 'γδ' in s:    lmap[v] = 'gdT'
        elif 'Plasma' in s or 'plasma' in s:     lmap[v] = 'PlasmaB'
        elif s == 'B':                           lmap[v] = 'B'
        elif 'NK' in s:                          lmap[v] = 'NK'
        elif 'Myeloid' in s or 'myeloid' in s:   lmap[v] = 'Myeloid'
        else:                                     lmap[v] = s
    adata.obs[LINEAGE_COL] = adata.obs['major_lineage'].map(lmap)

else:
    print("⚠️ No 'major_lineage' — extracting from subcluster names")
    def _lin(s):
        s = str(s)
        if s.startswith('CD4'):       return 'CD4T'
        elif s.startswith('CD8'):     return 'CD8T'
        elif s.startswith('gdT'):     return 'gdT'
        elif s.startswith('NK'):      return 'NK'
        elif s.startswith('B_'):      return 'B'
        elif s.startswith('plasma'):  return 'PlasmaB'
        elif any(s.startswith(x) for x in ['macro','mono','DC','dc','pDC','mast']):
            return 'Myeloid'
        else: return 'Unknown'
    adata.obs[LINEAGE_COL] = adata.obs[SUB_COL].apply(_lin)

adata.obs[LINEAGE_COL] = adata.obs[LINEAGE_COL].astype('category')

# Display
print(f"\n{'Lineage':<12} {'Cells':>10} {'%':>7}")
print("-" * 31)
total = len(adata)
for lin in sorted(adata.obs[LINEAGE_COL].unique()):
    n = (adata.obs[LINEAGE_COL] == lin).sum()
    print(f"{lin:<12} {n:>10,} {n/total*100:>6.1f}%")
print("-" * 31)
print(f"{'TOTAL':<12} {total:>10,}")

# 분석 대상 lineages (gdT 제외 — 781 cells, donor-level 통계 불가)
LINEAGES = ['Myeloid', 'CD4T', 'CD8T', 'NK', 'B', 'PlasmaB']
print(f"\n🎯 Analysis lineages (n=6): {LINEAGES}")
print(f"   gdT excluded ({(adata.obs[LINEAGE_COL]=='gdT').sum()} cells)")

# 각 lineage별 subcluster 목록
print(f"\n{'='*60}")
for lin in LINEAGES:
    mask = adata.obs[LINEAGE_COL] == lin
    subs = sorted(adata.obs.loc[mask, SUB_COL].unique())
    print(f"\n{lin} ({len(subs)} subclusters):")
    for s in subs:
        n = (adata.obs[SUB_COL] == s).sum()
        print(f"  {s}: {n:,}")

print("\n✅ Cell 6 complete")

✅ 'major_lineage' column exists — mapping to standard names
   Raw values: ['B', 'CD4_T', 'CD8_T', 'Myeloid', 'NK', 'PlasmaB', 'gdT']

Lineage           Cells       %
-------------------------------
B                21,325    8.8%
CD4T             69,446   28.6%
CD8T             70,065   28.8%
Myeloid          24,716   10.2%
NK               54,584   22.5%
PlasmaB           2,083    0.9%
gdT                 781    0.3%
-------------------------------
TOTAL           243,000

🎯 Analysis lineages (n=6): ['Myeloid', 'CD4T', 'CD8T', 'NK', 'B', 'PlasmaB']
   gdT excluded (781 cells)


Myeloid (11 subclusters):
  cDC1: 765
  cDC2-CLEC10A: 3,272
  cDC2-LAMP3: 341
  cDC2-MKI67: 408
  macro_c01-C1QC: 535
  macro_c02-NLRP3: 1,123
  macro_c03-FCGR3A: 827
  megakaryocyte: 1,634
  mono_c01-CD14: 10,099
  mono_c02-FCGR3A: 3,565
  pDC: 2,147

CD4T (10 subclusters):
  CD4T_c01-LEF1: 20,762
  CD4T_c02-ANXA2: 11,309
  CD4T_c03-GNLY: 6,822
  CD4T_c04-FOXP3: 5,777
  CD4T_c05-CCR7: 4,215
  CD4T_c06-IFI44L:

In [ ]:
# =================================================================
# CELL 7: Donor ID Extraction (이전 proportion 버그 수정 반영)
# =================================================================
# Format: 'GSM5519467_P190604_Blood_1' → split('_')[1] = 'P190604'
# NL:     'GSM5519479_D529074_Blood_1' → split('_')[1] = 'D529074'
# Special:'GSM5519483_Dhc570_Liver_1'  → split('_')[1] = 'Dhc570'

adata.obs['donor'] = (
    adata.obs[SAMPLE_COL]
    .astype(str)
    .str.split('_').str[1]
)

adata.obs['tissue'] = (
    adata.obs[SAMPLE_COL]
    .astype(str)
    .str.split('_').str[2]
)

# ---- 검증: hardcoded ground truth ----
EXPECTED = {
    'NL': sorted(['D528848','D529074','D529351','D529354','D529409','Dhc570']),
    'IT': sorted(['P190326','P190402','P190604','P190808','P190902','P190910']),
    'IA': sorted(['P190719','P190801','P190911','P191028','P191112']),
    'AR': sorted(['P190716','P191008','P191217']),
    'CR': sorted(['P191126','P191127','P191210']),
}

print("=" * 70)
print("  DONOR VERIFICATION")
print("=" * 70)

all_ok = True
for stage in ['NL','IT','IA','AR','CR']:
    mask = adata.obs[STAGE_COL] == stage
    found = sorted(adata.obs.loc[mask, 'donor'].unique())
    expected = EXPECTED[stage]
    match = (found == expected)
    status = "✅" if match else "❌"
    print(f"\n  {status} {stage} (n={len(found)}): {found}")
    if not match:
        print(f"     Expected: {expected}")
        all_ok = False

# NaN check
n_nan = adata.obs['donor'].isna().sum()
print(f"\n  Donor NaN: {n_nan}")

n_donors = adata.obs['donor'].nunique()
print(f"  Total donors: {n_donors}")

# Tissue check
print(f"\n  Tissue: {adata.obs['tissue'].value_counts().to_dict()}")

if all_ok and n_nan == 0 and n_donors == 23:
    print(f"\n  ✅ ALL 23 DONORS VERIFIED (NL=6, IT=6, IA=5, AR=3, CR=3)")
else:
    print(f"\n  ⚠️ DONOR VERIFICATION ISSUE — check above")

print("✅ Cell 7 complete")


  DONOR VERIFICATION

  ✅ NL (n=6): ['D528848', 'D529074', 'D529351', 'D529354', 'D529409', 'Dhc570']

  ✅ IT (n=6): ['P190326', 'P190402', 'P190604', 'P190808', 'P190902', 'P190910']

  ✅ IA (n=5): ['P190719', 'P190801', 'P190911', 'P191028', 'P191112']

  ✅ AR (n=3): ['P190716', 'P191008', 'P191217']

  ✅ CR (n=3): ['P191126', 'P191127', 'P191210']

  Donor NaN: 0
  Total donors: 23

  Tissue: {'Blood': 136408, 'Liver': 106592}

  ✅ ALL 23 DONORS VERIFIED (NL=6, IT=6, IA=5, AR=3, CR=3)
✅ Cell 7 complete


In [ ]:
# =================================================================
# CELL 8: Study Group Configuration + Color Scheme
# =================================================================

# ---- Study group structure ----
# Chronic HBV disease stages: IT → IA → CR (vertical/childhood infection)
# Independent comparator: AR (adult-onset acute hepatitis B, self-limited)
# Healthy control: NL
# 논문에서 "disease stage"는 IT→IA→CR 만 해당
# 전체 5개 = "five study groups" or "five donor groups"

STAGE_ORDER = ['NL', 'IT', 'IA', 'AR', 'CR']
CHRONIC_STAGES = ['IT', 'IA', 'CR']  # disease spectrum
DONOR_N = {'NL': 6, 'IT': 6, 'IA': 5, 'AR': 3, 'CR': 3}

# ---- Consistent colors ----
STAGE_COLORS = {
    'NL': '#2ECC71',   # green
    'IT': '#E74C3C',   # red
    'IA': '#F39C12',   # orange
    'AR': '#3498DB',   # blue
    'CR': '#9B59B6',   # purple
}

LINEAGE_COLORS = {
    'Myeloid':  '#E74C3C',
    'CD4T':     '#2ECC71',
    'CD8T':     '#9B59B6',
    'NK':       '#F39C12',
    'B':        '#3498DB',
    'PlasmaB':  '#8B4513',
    'gdT':      '#FF69B4',
}

# ---- Matplotlib defaults ----
plt.rcParams.update({
    'figure.facecolor': 'white',
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 100,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
})

print("📋 Study Group Configuration:")
print(f"   All groups (5):       {STAGE_ORDER}")
print(f"   Chronic spectrum (3): {' → '.join(CHRONIC_STAGES)}")
print(f"   Comparator:           AR (acute self-limited hepatitis B)")
print(f"   Control:              NL (healthy)")
print()
for s in STAGE_ORDER:
    n_d = DONOR_N[s]
    n_c = (adata.obs[STAGE_COL] == s).sum()
    print(f"   {s}: n={n_d} donors, {n_c:>6,} cells")

print("\n✅ Cell 8 complete")

📋 Study Group Configuration:
   All groups (5):       ['NL', 'IT', 'IA', 'AR', 'CR']
   Chronic spectrum (3): IT → IA → CR
   Comparator:           AR (acute self-limited hepatitis B)
   Control:              NL (healthy)

   NL: n=6 donors, 42,579 cells
   IT: n=6 donors, 49,179 cells
   IA: n=5 donors, 62,545 cells
   AR: n=3 donors, 45,452 cells
   CR: n=3 donors, 43,245 cells

✅ Cell 8 complete


In [ ]:
# =================================================================
# CELL 9: Utility Functions (Donor-Level Statistics)
# =================================================================

def get_gene_expr(adata_sub, gene):
    """유전자 발현량 추출 (sparse matrix 처리)"""
    if gene not in adata_sub.var_names:
        return None
    idx = list(adata_sub.var_names).index(gene)
    x = adata_sub.X[:, idx]
    if issparse(x):
        return np.asarray(x.todense()).flatten().astype(np.float64)
    return np.asarray(x).flatten().astype(np.float64)


def donor_means(adata_sub, gene):
    """Donor-level mean expression → DataFrame [donor, stage, mean_expr]"""
    expr = get_gene_expr(adata_sub, gene)
    if expr is None:
        return None
    df = pd.DataFrame({
        'expr': expr,
        'donor': adata_sub.obs['donor'].values,
        'stage': adata_sub.obs[STAGE_COL].values,
    })
    return df.groupby(['donor','stage'])['expr'].mean().reset_index(
        ).rename(columns={'expr': 'mean_expr'})


def mwu(donor_df, g1='NL', g2='IT'):
    """Mann-Whitney U at donor level. Returns dict or None."""
    v1 = donor_df[donor_df['stage']==g1]['mean_expr'].values
    v2 = donor_df[donor_df['stage']==g2]['mean_expr'].values
    if len(v1) < 2 or len(v2) < 2:
        return None
    stat, p = stats.mannwhitneyu(v1, v2, alternative='two-sided')
    m1, m2 = np.mean(v1), np.mean(v2)
    fc = ((m2 - m1) / m1 * 100) if m1 > 0 else (np.inf if m2 > 0 else 0.0)
    # Consistency: donor pairs where g2 > g1
    n_pairs = len(v1) * len(v2)
    consist = int(np.sum(v2[:, None] > v1[None, :]))
    return {
        'g1': g1, 'g2': g2,
        'n1': len(v1), 'n2': len(v2),
        'mean1': m1, 'mean2': m2,
        'pct_change': fc,
        'consist': consist, 'n_pairs': n_pairs,
        'consist_str': f"{consist}/{n_pairs}",
        'p': p, 'sig': p < 0.05,
    }


def screen_gene(adata_lin, gene, ref='NL'):
    """한 gene을 ref 대비 모든 group과 비교"""
    dm = donor_means(adata_lin, gene)
    if dm is None:
        return []
    results = []
    for tgt in STAGE_ORDER:
        if tgt == ref:
            continue
        r = mwu(dm, g1=ref, g2=tgt)
        if r:
            r['gene'] = gene
            r['lineage'] = ''  # caller가 설정
            results.append(r)
    return results


def donor_proportion(adata, lineage_col, stage_col, donor_col='donor'):
    """각 donor 내 lineage 비율(%) → DataFrame [donor, stage, lineage, pct]
    C1 proportion 분석의 핵심 함수.
    """
    # donor별 total cell count
    donor_total = adata.obs.groupby(donor_col).size().rename('total')

    # donor × lineage cell count
    ct = adata.obs.groupby([donor_col, lineage_col]).size().rename('count').reset_index()
    ct = ct.merge(donor_total, on=donor_col)
    ct['pct'] = ct['count'] / ct['total'] * 100

    # stage 추가
    donor_stage = adata.obs.groupby(donor_col)[stage_col].first()
    ct = ct.merge(donor_stage, on=donor_col)

    return ct


def save_fig(fig, name, subdir='C1_proportion'):
    """PNG 저장"""
    path = f'{OUT}/{subdir}/{name}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"💾 {path}")
    return path


print("✅ Cell 9 complete — utility functions defined")
print("   get_gene_expr / donor_means / mwu / screen_gene")
print("   donor_proportion / save_fig")



✅ Cell 9 complete — utility functions defined
   get_gene_expr / donor_means / mwu / screen_gene
   donor_proportion / save_fig


In [ ]:
# =================================================================
# CELL 10: Final Data Integrity Check
# =================================================================
print("=" * 70)
print("  DATA INTEGRITY SUMMARY — Option A Bottom-Up Analysis")
print("=" * 70)

print(f"\n  Dataset:     GSE182159 (Zhang et al., Gut 2022)")
print(f"  Cells:       {adata.shape[0]:,}")
print(f"  Genes:       {adata.shape[1]:,}")
print(f"  Subclusters: {adata.obs[SUB_COL].nunique()}")
print(f"  Lineages:    {adata.obs[LINEAGE_COL].nunique()} (6 for analysis + gdT)")
print(f"  Donors:      {adata.obs['donor'].nunique()}")

print(f"\n  {'Group':<6} {'n':>4} {'Cells':>10} {'%':>7}")
print(f"  {'-'*30}")
for s in STAGE_ORDER:
    n_d = DONOR_N[s]
    n_c = (adata.obs[STAGE_COL] == s).sum()
    print(f"  {s:<6} {n_d:>4} {n_c:>10,} {n_c/total*100:>6.1f}%")

print(f"\n  Key columns:")
print(f"    STAGE_COL  = '{STAGE_COL}'")
print(f"    SUB_COL    = '{SUB_COL}'")
print(f"    LINEAGE_COL= '{LINEAGE_COL}'")
print(f"    SAMPLE_COL = '{SAMPLE_COL}'")
print(f"    donor, tissue = extracted from {SAMPLE_COL}")

# AUCell 존재 여부
pw_cols = [c for c in adata.obs.columns if 'PW_' in c or 'AUCell' in c or 'aucell' in c]
if pw_cols:
    print(f"\n  ⚠️ Pre-existing pathway columns ({len(pw_cols)}):")
    for c in pw_cols:
        print(f"    {c}")
    print("  → 이들은 이전 분석 잔여물. Option A에서는 fresh compute.")
else:
    print(f"\n  ℹ️ No pre-computed pathway scores — fresh AUCell needed for C2")

print(f"\n{'=' * 70}")
print(f"  ✅ SETUP COMPLETE — Ready for C1: Lineage Proportion Analysis")
print(f"  Output: {OUT}")
print(f"{'=' * 70}")

  DATA INTEGRITY SUMMARY — Option A Bottom-Up Analysis

  Dataset:     GSE182159 (Zhang et al., Gut 2022)
  Cells:       243,000
  Genes:       24,452
  Subclusters: 59
  Lineages:    7 (6 for analysis + gdT)
  Donors:      23

  Group     n      Cells       %
  ------------------------------
  NL        6     42,579   17.5%
  IT        6     49,179   20.2%
  IA        5     62,545   25.7%
  AR        3     45,452   18.7%
  CR        3     43,245   17.8%

  Key columns:
    STAGE_COL  = 'Stage'
    SUB_COL    = 'gut2021_subcluster_v2'
    LINEAGE_COL= 'lineage'
    SAMPLE_COL = 'sample'
    donor, tissue = extracted from sample

  ⚠️ Pre-existing pathway columns (6):
    PW_mTOR_signaling
    PW_glycolysis
    PW_oxidative_phosphorylation
    PW_nk_cell_cytotoxicity
    PW_il15_signaling
    PW_b_cell_differentiation
  → 이들은 이전 분석 잔여물. Option A에서는 fresh compute.

  ✅ SETUP COMPLETE — Ready for C1: Lineage Proportion Analysis
  Output: /content/drive/MyDrive/ITLAS/results/optionA_26Mar0

In [ ]:
# ============================================================
# CELL 2: Load Data & Detect Columns
# ============================================================
print("Loading h5ad file...")
adata = sc.read_h5ad(DATA_PATH)
print(f"Loaded: {adata.shape[0]} cells × {adata.shape[1]} genes")

# --- Print ALL columns for reference ---
print("\n" + "="*60)
print("ALL OBS COLUMNS:")
print("="*60)
for i, col in enumerate(adata.obs.columns):
    print(f"  [{i:2d}] '{col}' ({adata.obs[col].dtype}) — {adata.obs[col].nunique()} unique")

print("\nSample rows:")
print(adata.obs.head(3).to_string())

# --- Auto-detect stage column (contains NL/IT/IA or AC) ---
stage_col = None
for col in adata.obs.columns:
    try:
        uniq = set(adata.obs[col].dropna().astype(str).unique())
        if 'IT' in uniq and ('NL' in uniq or 'IA' in uniq):
            stage_col = col
            break
    except:
        continue

# --- Auto-detect lineage column (contains Myeloid/CD4T/NK) ---
lineage_col = None
for col in adata.obs.columns:
    try:
        uniq = set(adata.obs[col].dropna().astype(str).unique())
        if 'Myeloid' in uniq or 'CD4T' in uniq or 'NK' in uniq:
            lineage_col = col
            break
    except:
        continue

# --- Auto-detect donor column (3-23 unique, NOT stage/lineage) ---
donor_col = None
# Priority order: sample_id first
for col in ['sample_id', 'donor_id', 'donor', 'patient', 'sample']:
    if col in adata.obs.columns:
        donor_col = col
        break
# If not found by name, find by unique count
if donor_col is None:
    for col in adata.obs.columns:
        if col == stage_col or col == lineage_col:
            continue
        n = adata.obs[col].nunique()
        if 10 <= n <= 30:  # expect ~23 donors
            donor_col = col
            break

# --- Report results ---
print("\n" + "="*60)
print("DETECTED COLUMNS:")
print("="*60)

if stage_col:
    print(f"  ✅ Stage:   '{stage_col}' → {sorted(adata.obs[stage_col].astype(str).unique().tolist())}")
    # AC → CR rename
    if 'AC' in adata.obs[stage_col].values:
        print("     ⚠️ Found 'AC' — renaming to 'CR'")
        adata.obs[stage_col] = adata.obs[stage_col].replace({'AC': 'CR'})
        print(f"     After fix: {sorted(adata.obs[stage_col].astype(str).unique().tolist())}")
else:
    print("  ❌ Stage column NOT FOUND!")
    print("     >>> Set manually: stage_col = 'YOUR_COLUMN_NAME' <<<")

if lineage_col:
    print(f"  ✅ Lineage: '{lineage_col}' → {sorted(adata.obs[lineage_col].astype(str).unique().tolist())}")
else:
    print("  ❌ Lineage column NOT FOUND!")
    print("     >>> Set manually: lineage_col = 'YOUR_COLUMN_NAME' <<<")

if donor_col:
    print(f"  ✅ Donor:   '{donor_col}' → {adata.obs[donor_col].nunique()} unique donors")
else:
    print("  ❌ Donor column NOT FOUND!")
    print("     >>> Set manually: donor_col = 'YOUR_COLUMN_NAME' <<<")

# --- Verify donor counts per stage ---
if stage_col and donor_col:
    print(f"\n  Donors per stage:")
    for stage in ['NL', 'IT', 'IA', 'AR', 'CR']:
        mask = adata.obs[stage_col].astype(str) == stage
        if mask.sum() > 0:
            n_donors = adata.obs.loc[mask, donor_col].nunique()
            n_cells = mask.sum()
            print(f"    {stage}: {n_donors} donors, {n_cells:,} cells")
        else:
            print(f"    {stage}: not found")
    print("\n  Expected: NL≈6, IT≈6, IA≈5, AR≈3, CR≈3")

# --- STOP here if any column missing ---
if not all([stage_col, lineage_col, donor_col]):
    raise ValueError(
        "Column detection failed. Set manually BELOW this cell:\n"
        "  stage_col   = '...'\n"
        "  lineage_col = '...'\n"
        "  donor_col   = '...'\n"
        "Then re-run from Cell 3."
    )

print("\n✅ All columns detected. Proceed to Cell 3.")

# --- Update LINEAGES to match actual data ---
actual_lineages = sorted(adata.obs[lineage_col].unique().tolist())
print(f"\nActual lineages in data: {actual_lineages}")
print(f"Original LINEAGES list:  {LINEAGES}")

# Update LINEAGES to only include what exists in data
# Also handle name variants (e.g., CD4_T vs CD4T)
LINEAGES = [l for l in actual_lineages if l in LINEAGES]
extra = [l for l in actual_lineages if l not in LINEAGES and l != 'gdT']
if extra:
    print(f"  ⚠️ Extra lineages not in original list: {extra}")
    print(f"     These may be name variants (CD4_T vs CD4T). Adding them.")
    LINEAGES.extend(extra)
missing = [l for l in ['Myeloid', 'CD4T', 'CD8T', 'NK', 'B', 'PlasmaB'] if l not in LINEAGES]
if missing:
    print(f"  ⚠️ Expected lineages not found: {missing}")
    print(f"     Check if names differ (e.g., 'CD4_T' vs 'CD4T')")

print(f"\nFinal LINEAGES for analysis: {LINEAGES}")

Loading h5ad file...
Loaded: 243000 cells × 24452 genes

ALL OBS COLUMNS:
  [ 0] 'sample' (category) — 46 unique
  [ 1] 'tissue' (category) — 2 unique
  [ 2] 'Stage' (category) — 5 unique
  [ 3] 'IT_cluster_21' (float64) — 242960 unique
  [ 4] 'IT_cluster_23' (float64) — 240984 unique
  [ 5] 'IT_cluster_25' (float64) — 114299 unique
  [ 6] 'IT_nk_collapse' (float32) — 168875 unique
  [ 7] 'IT_IT_signature' (float64) — 243000 unique
  [ 8] 'GSM_ID' (category) — 46 unique
  [ 9] 'IT_score_v2' (float64) — 243000 unique
  [10] 'IT_score_v3' (float64) — 243000 unique
  [11] 'IT_score_v4' (float64) — 242998 unique
  [12] 'IT_signature_final' (float64) — 242998 unique
  [13] 'IT_like' (category) — 2 unique
  [14] 'PW_mTOR_signaling' (float32) — 30497 unique
  [15] 'PW_glycolysis' (float32) — 208005 unique
  [16] 'PW_oxidative_phosphorylation' (float32) — 237150 unique
  [17] 'PW_nk_cell_cytotoxicity' (float32) — 137083 unique
  [18] 'PW_il15_signaling' (float32) — 93432 unique
  [19] 'PW_b_ce